# Experiment 02: Regression Analysis (Linear, Ridge & Lasso)
**Aim:** Predict Uber ride fares using Linear, Ridge, and Lasso Regression and compare model performances.

In [4]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Generate sample dataset if file not present
if not os.path.exists('uber_rides.csv'):
    np.random.seed(42)
    pd.DataFrame({
        'fare_amount': np.random.uniform(5, 50, 500),
        'pickup_datetime': pd.date_range('2025-01-01', periods=500, freq='h'),
        'pickup_longitude': -73.98 + np.random.normal(0, 0.02, 500),
        'pickup_latitude': 40.75 + np.random.normal(0, 0.02, 500),
        'dropoff_longitude': -73.98 + np.random.normal(0, 0.03, 500),
        'dropoff_latitude': 40.75 + np.random.normal(0, 0.03, 500),
        'passenger_count': np.random.randint(1, 6, 500)
    }).to_csv('uber_rides.csv', index=False)

# 1. Load & Feature Engineering (Haversine Distance)
df = pd.read_csv('uber_rides.csv').dropna().reset_index(drop=True)
df = df[(df['fare_amount'] >= 2.5) & (df['fare_amount'] <= 150.0)]

lon1, lat1, lon2, lat2 = map(np.radians, [df['pickup_longitude'], df['pickup_latitude'], df['dropoff_longitude'], df['dropoff_latitude']])
a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
df['distance_km'] = (6371 * 2 * np.arcsin(np.sqrt(a))).round(2)

dt = pd.to_datetime(df['pickup_datetime'])
df['pickup_hour'] = dt.dt.hour
df['day_of_week'] = dt.dt.dayofweek
df = df[(df['distance_km'] >= 0.1) & (df['distance_km'] <= 50.0)]
df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,distance_km,pickup_hour,day_of_week
0,21.854305,2025-01-01 00:00:00,-73.973165,40.761679,-73.940948,40.725712,1,4.83,0,2
1,47.782144,2025-01-01 01:00:00,-73.942477,40.742814,-73.933155,40.762722,2,2.35,1,2
2,37.939727,2025-01-01 02:00:00,-73.960992,40.761813,-73.979040,40.735785,2,3.27,2,2
3,31.939632,2025-01-01 03:00:00,-73.991538,40.772174,-74.002603,40.749566,5,2.68,3,2
4,12.020839,2025-01-01 04:00:00,-73.997968,40.766410,-73.966201,40.766389,4,2.68,4,2


In [5]:
# 2. Train Models & Evaluate Performance
features = ['distance_km', 'passenger_count', 'pickup_hour', 'day_of_week']
X = StandardScaler().fit_transform(df[features])
y = df['fare_amount']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=0.1)
}

results = []
for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    results.append({
        'Model': name,
        'MAE': round(mean_absolute_error(y_te, pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_te, pred)), 4),
        'R2 Score': round(r2_score(y_te, pred), 4)
    })

comparison_df = pd.DataFrame(results)
print("=== Model Performance Comparison ===")
comparison_df

=== Model Performance Comparison ===


,Model,MAE,RMSE,R2 Score
0,Linear Regression,11.3080,13.0915,0.0100
1,Ridge (L2),11.3083,13.0916,0.0100
2,Lasso (L1),11.3210,13.0949,0.0095
